# Domain 1: Agentic Architecture & Orchestration
## Task Statement 1.1: Design and Implement Agentic Loops
**Certification Weight**: 27% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to build autonomous agentic loops using the Claude API, inspect `response.stop_reason` deterministically, maintain conversational history with tool results, and avoid completion anti-patterns.

---

## 🏗️ Architecture: The Deterministic Loop Lifecycle

```mermaid
flowchart TD
    Start["User Prompt Injected"] --> SendRequest["Call Claude Messages API<br>(model: claude-sonnet-4-5)"]
    SendRequest --> CheckStop{"Inspect stop_reason"}
    
    CheckStop -- "tool_use" --> AppendAssistant["1. Append Assistant Turn<br>(Include thinking & tool_use blocks)"]
    AppendAssistant --> ExecLocal["2. Execute Requested Tools Locally"]
    ExecLocal --> AppendUser["3. Append User Turn<br>(type: tool_result blocks)"]
    AppendUser --> SendRequest
    
    CheckStop -- "end_turn" --> Terminate["4. Task Complete<br>Extract & Return Final Text"]
    CheckStop -- "max_tokens / error" --> HandleException["Safety Exception Handler"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Event / Strategy | Implementation | Exam Trap to Reject |
|---|---|---|
| **Loop Termination Signal** | `response.stop_reason == "end_turn"` | **TRAP**: Parsing text for words like `"Done"`, `"Completed"`, or using regex. |
| **Tool Execution Round-Trip** | Append assistant `tool_use` message, then append user `tool_result` message. | **TRAP**: Appending only the tool result without the assistant's preceding `tool_use` turn. |
| **Thinking Block Continuity** | Echo assistant `thinking` blocks back **unmodified with signatures** in subsequent turns. | **TRAP**: Stripping thinking blocks to save tokens (Anthropic API rejects with `400 Bad Request`). |
| **Safety Limits** | `MAX_ITERATIONS = 15` as an emergency runaway brake. | **TRAP**: Relying on turn count or iteration caps as normal termination logic. |

---

## 💡 Architectural Decision Matrix

| Question | Recommended Choice | Rationale |
|---|---|---|
| When should the agent stop calling tools? | Only when `stop_reason == "end_turn"`. | The model alone decides when it has accumulated sufficient evidence to answer. |
| How to handle sampling when thinking is on? | Omit `temperature` completely. | Custom temperature parameters trigger an HTTP `400 Bad Request` API error. |
| Which model to use for the orchestrator loop? | `claude-sonnet-4-5` | Best balance of deep multi-step reasoning, tool choice accuracy, and low latency. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Offline Simulation Setup
import ast
import json
import operator
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

# Dynamically locate repository root
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import mock client for 100% offline, zero-cost execution
from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse
from lib.response_logger import ResponseTrail

print(" Repository root loaded:", repo_root)
print(" Offline Mock Claude Client ready (Zero API calls, Zero tokens)")

### 🛠️ Step 2: Safe Tool Implementations & Schemas
We implement two production-grade tools:
1. `calculator`: Evaluates arithmetic expressions safely using Python's `ast` module (avoiding unsafe `eval()`).
2. `web_search`: Simulates targeted knowledge retrieval with pre-indexed fact snippets.

In [ ]:
# Step 2: Tool Logic & JSON Schemas
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def safe_eval_expr(node: ast.AST) -> float:
    """Safely evaluate an arithmetic AST expression without eval()."""
    if isinstance(node, ast.Expression):
        return safe_eval_expr(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](safe_eval_expr(node.left), safe_eval_expr(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](safe_eval_expr(node.operand))
    raise ValueError(f"Unsupported AST node: {type(node).__name__}")

def calculator_tool(expression: str) -> Dict[str, Any]:
    """Execute mathematical expressions safely."""
    try:
        parsed = ast.parse(expression.strip(), mode="eval")
        result = safe_eval_expr(parsed)
        return {"result": int(result) if result.is_integer() else result}
    except Exception as exc:
        return {"isError": True, "error": f"Evaluation error: {exc}"}

def web_search_tool(query: str) -> Dict[str, Any]:
    """Simulate search retrieval from trusted reference data."""
    kb = {
        "speed of light": "The speed of light in vacuum is approximately 299,792,458 meters per second.",
        "tokyo population": "The population of Tokyo metropolitan area is approximately 37,400,000.",
    }
    q = query.lower()
    for key, val in kb.items():
        if key in q or any(w in q for w in key.split()):
            return {"found": True, "results": [{"title": key.title(), "snippet": val}]}
    return {"found": False, "results": [], "message": f"No entries for: {query}"}

TOOLS = [
    {
        "name": "calculator",
        "description": "Calculates the result of a mathematical expression. Input must be arithmetic only.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Arithmetic expression, e.g. '234 * 18'"}},
            "required": ["expression"],
        },
    },
    {
        "name": "web_search",
        "description": "Searches reference documentation and facts by search query.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search term or question"}},
            "required": ["query"],
        },
    },
]

def execute_tool(tool_name: str, tool_inputs: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name == "calculator":
        return calculator_tool(tool_inputs.get("expression", ""))
    elif tool_name == "web_search":
        return web_search_tool(tool_inputs.get("query", ""))
    return {"isError": True, "error": f"Unknown tool: {tool_name}"}

print(" Tools and schemas initialized successfully.")

### 🔄 Step 3: The Deterministic Agentic Loop Implementation
The core loop orchestrates conversational turns by inspecting `response.stop_reason`.
- If `stop_reason == "tool_use"`, we execute each tool and feed `tool_result` blocks back.
- If `stop_reason == "end_turn"`, the agent has completed reasoning and returns the answer.

In [ ]:
# Step 3: The Core Autonomous Agent Loop
def run_agentic_loop(prompt: str, client: Any, trail: Optional[ResponseTrail] = None, max_iterations: int = 15) -> str:
    """
    Executes the standard agentic loop with stop_reason lifecycle management.
    """
    messages: List[Dict[str, Any]] = [{"role": "user", "content": prompt}]
    session_id = "sess_sim_01"
    iteration = 0

    print(f"\n=== Starting Agentic Loop for prompt: '{prompt}' ===")

    while iteration < max_iterations:
        iteration += 1
        print(f"\n[Turn {iteration}] Calling Claude API...")

        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            messages=messages,
            tools=TOOLS,
        )

        stop_reason = response.stop_reason
        print(f"[Turn {iteration}] stop_reason: '{stop_reason}'")

        # Check if model wants to invoke tools
        if stop_reason == "tool_use":
            # 1. Append assistant turn with raw tool_use blocks
            assistant_content = [b.model_dump() for b in response.content]
            messages.append({"role": "assistant", "content": assistant_content})

            # 2. Execute tools and collect tool_result blocks
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  -> Executing Tool: {block.name}({block.input})")
                    res = execute_tool(block.name, block.input)
                    print(f"  <- Tool Result: {res}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(res),
                    })

            # 3. Append user turn containing tool results
            messages.append({"role": "user", "content": tool_results})
            continue

        # Deterministic exit signal
        if stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if b.type == "text")
            print(f"\n[Turn {iteration}] Task Complete. Final Response: {final_text}")
            return final_text

        raise RuntimeError(f"Unexpected stop_reason: {stop_reason}")

    raise TimeoutError(f"Safety limit of {max_iterations} iterations reached.")

### 🧪 Step 4: Multi-Turn Offline Simulation
We simulate a multi-turn conversation without any live API costs:
- **Turn 1**: Claude decides to invoke `calculator` for `125 * 8`.
- **Turn 2**: Claude receives the calculation `1000` and emits `stop_reason == "end_turn"` with the final text.

In [ ]:
# Step 4: Execute Simulation with MockClaudeClient
# Queue Turn 1: tool_use request
turn1_response = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="tool_use",
            name="calculator",
            tool_id="call_calc_981",
            input_data={"expression": "125 * 8"},
        )
    ],
    stop_reason="tool_use",
)

# Queue Turn 2: end_turn conclusion
turn2_response = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="text",
            text="125 multiplied by 8 equals 1,000.",
        )
    ],
    stop_reason="end_turn",
)

mock_client = MockClaudeClient([turn1_response, turn2_response])
final_output = run_agentic_loop("What is 125 * 8?", mock_client)
print("\nResult verified:", final_output)

### 🔬 Step 5: Automated Verification & Edge Case Checks
We verify that:
1. Arithmetic evaluations are completely protected against injection attacks.
2. The agent loop correctly handles single-turn and multi-turn stops.
3. The conversation history correctly interleaves `tool_use` and `tool_result`.

In [ ]:
# Step 5: Verification Assertions
# 1. Verify AST calculator safety against malicious code injection
safe_res = calculator_tool("5 * 20 + 10")
assert safe_res["result"] == 110, f"Expected 110, got {safe_res}"

injection_attempt = calculator_tool("__import__('os').system('ls')")
assert injection_attempt.get("isError") is True, "Safety filter must reject eval injection!"

# 2. Verify web search functionality
search_res = web_search_tool("speed of light")
assert search_res["found"] is True
assert "299,792,458" in search_res["results"][0]["snippet"]

# 3. Verify final output
assert "1,000" in final_output or "1000" in final_output

print(" All verification assertions passed successfully! 100% offline verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Inspect `response.stop_reason` exclusively. Never inspect response text string patterns to determine if an agent is done.
- **Rule 2**: Both `tool_use` and `tool_result` must be appended in distinct conversational roles (`assistant` then `user`).
- **Rule 3**: `MAX_ITERATIONS` is a fallback safeguard, never the primary stopping mechanism.